In [1]:
import pandas as pd
import requests
import numpy as np

In [2]:
# URLs for All-NBA Teams
urls = {
    "1st Team": "https://www.espn.com/nba/history/awards/_/id/44",
    "2nd Team": "https://www.espn.com/nba/history/awards/_/id/45",
    "3rd Team": "https://www.espn.com/nba/history/awards/_/id/46"
}

In [3]:
all_data = []

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

for team_type, url in urls.items():
    print(f"Scraping {team_type} from {url}...")
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            dfs = pd.read_html(response.content)
            print(f"  Found {len(dfs)} tables.")
            
            target_df = None
            # Heuristic: The table we want is likely the largest one (most rows)
            if dfs:
                target_df = max(dfs, key=lambda x: x.shape[0])
                print(f"  Largest table shape: {target_df.shape}")

            if target_df is not None and target_df.shape[0] > 5:
                # Fix headers
                # Check if first row looks like headers (contains 'Year' or 'Player')
                header_row_idx = None
                for i in range(min(5, len(target_df))):
                    row_str = target_df.iloc[i].astype(str).str.upper().tolist()
                    if 'PLAYER' in row_str or 'YEAR' in row_str or 'SEASON' in row_str:
                        header_row_idx = i
                        break
                
                if header_row_idx is not None:
                    # Set header
                    target_df.columns = target_df.iloc[header_row_idx]
                    target_df = target_df[header_row_idx + 1:]
                
                # Normalize columns
                target_df.columns = [str(c).upper().strip() for c in target_df.columns]
                
                if 'PLAYER' in target_df.columns:
                    # Filter out repeated header rows
                    target_df = target_df[target_df['PLAYER'] != 'PLAYER']
                    
                    # Handle Year Column
                    year_col = 'YEAR' if 'YEAR' in target_df.columns else 'SEASON' if 'SEASON' in target_df.columns else None
                    
                    if year_col:
                        # Forward fill year - replacing empty strings or nulls
                        target_df[year_col] = target_df[year_col].replace('', np.nan).ffill()
                        
                        # Select columns
                        cols_to_keep = [year_col, 'PLAYER']
                        if 'POS' in target_df.columns:
                            cols_to_keep.append('POS')
                        if 'TEAM' in target_df.columns:
                            cols_to_keep.append('TEAM') # Capture Team
                        
                        subset = target_df[cols_to_keep].copy()
                        subset = subset.rename(columns={year_col: 'Year', 'POS': 'Position', 'PLAYER': 'Player', 'TEAM': 'Team'})
                        # Removed Team_Type addition as requested
                        # subset['Team_Type'] = team_type 
                        
                        all_data.append(subset)
                    else:
                         print(f"  Could not find Year/Season column in table for {team_type}")
                else:
                    print(f"  Could not find 'PLAYER' column. Columns found: {target_df.columns.tolist()}")

            else:
                print(f"  No suitable large table found for {team_type}")
        else:
            print(f"Failed to fetch {url}: {response.status_code}")
            
    except Exception as e:
        print(f"Error scraping {team_type}: {e}")

# Combine and Process
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    
    def clean_year(val):
        try:
            val = str(val).strip()
            # Handle "2023-24" or just "2025"
            if '-' in val:
                 parts = val.split('-')
                 if len(parts[0]) == 4:
                     return int(parts[0]) + 1 if len(parts[1]) > 0 else int(parts[0])
            return int(float(val))
        except:
            return None

    final_df['Year'] = final_df['Year'].apply(clean_year)
    
    # Drop rows without player or year
    final_df = final_df.dropna(subset=['Player', 'Year'])
    final_df['Year'] = final_df['Year'].astype(int)
    
    # FILTER: Only keep years 2015-2026
    final_df = final_df[(final_df['Year'] >= 2015) & (final_df['Year'] <= 2026)]
    
    # Sort
    # Fixed ValueError: Length of ascending (2) == length of by (2)
    final_df = final_df.sort_values(by=['Year', 'Player'], ascending=[False, True])
    
    # Final Selection
    cols = ['Year', 'Player', 'Team', 'Position'] 
    
    # Ensure columns exist
    if 'Position' not in final_df.columns:
        final_df['Position'] = 'Unknown'
    if 'Team' not in final_df.columns:
        final_df['Team'] = 'Unknown'
    
    df = final_df[cols]
    
    print("\nScraping Complete!")
    print(f"Total rows: {len(df)}")
    print(df.head(10))
    
    try:
        df.to_csv('data/nba_all_stars.csv', index=False)
        print("Saved to data/nba_all_stars.csv")
    except PermissionError:
        print("ERROR: Could not save to data/nba_all_stars.csv because it is open in another program.")
        print("Please close the file and run this cell again.")
    except Exception as e:
        print(f"Error saving file: {e}")
    
else:
    print("No data collected.")

Scraping 1st Team from https://www.espn.com/nba/history/awards/_/id/44...
  Found 1 tables.
  Largest table shape: (398, 10)
Scraping 2nd Team from https://www.espn.com/nba/history/awards/_/id/45...
  Found 1 tables.
  Largest table shape: (396, 10)
Scraping 3rd Team from https://www.espn.com/nba/history/awards/_/id/46...
  Found 1 tables.
  Largest table shape: (187, 10)

Scraping Complete!
Total rows: 165
     Year                 Player                    Team Position
400  2025        Anthony Edwards  Minnesota Timberwolves       SG
793  2025        Cade Cunningham         Detroit Pistons       PG
2    2025       Donovan Mitchell     Cleveland Cavaliers       SG
399  2025            Evan Mobley     Cleveland Cavaliers       PF
0    2025  Giannis Antetokounmpo         Milwaukee Bucks       PF
398  2025          Jalen Brunson         New York Knicks       PG
794  2025         Jalen Williams   Oklahoma City Thunder        F
790  2025           James Harden             LA Clippers     

In [ ]:
# Manually add 2026 All-Stars (sourced from https://www.nba.com/allstar/2026/roster)
# Note: Team affiliations are based on the roster page provided.
# Positions updated to specific roles (PG, SG, SF, PF, C) based on general player profiles.

all_stars_2026_data = [
    # USA STARS
    {"Player": "Scottie Barnes", "Team": "Toronto Raptors", "Position": "SF"},
    {"Player": "Devin Booker", "Team": "Phoenix Suns", "Position": "SG"},
    {"Player": "Cade Cunningham", "Team": "Detroit Pistons", "Position": "PG"},
    {"Player": "Jalen Duren", "Team": "Detroit Pistons", "Position": "C"},
    {"Player": "Anthony Edwards", "Team": "Minnesota Timberwolves", "Position": "SG"},
    {"Player": "Chet Holmgren", "Team": "Oklahoma City Thunder", "Position": "C"},
    {"Player": "Jalen Johnson", "Team": "Atlanta Hawks", "Position": "PF"},
    {"Player": "Tyrese Maxey", "Team": "Philadelphia 76ers", "Position": "PG"},
    
    # USA STRIPES
    {"Player": "Jaylen Brown", "Team": "Boston Celtics", "Position": "SG"},
    {"Player": "Jalen Brunson", "Team": "New York Knicks", "Position": "PG"},
    {"Player": "Kevin Durant", "Team": "Houston Rockets", "Position": "PF"},
    {"Player": "De'Aaron Fox", "Team": "San Antonio Spurs", "Position": "PG"},
    {"Player": "Brandon Ingram", "Team": "Toronto Raptors", "Position": "SF"},
    {"Player": "LeBron James", "Team": "Los Angeles Lakers", "Position": "SF"},
    {"Player": "Kawhi Leonard", "Team": "LA Clippers", "Position": "SF"},
    {"Player": "Donovan Mitchell", "Team": "Cleveland Cavaliers", "Position": "SG"},
    {"Player": "Stephen Curry", "Team": "Golden State Warriors", "Position": "PG"},
    
    # WORLD
    {"Player": "Deni Avdija", "Team": "Portland Trail Blazers", "Position": "SF"},
    {"Player": "Luka Doncic", "Team": "Los Angeles Lakers", "Position": "PG"},
    {"Player": "Shai Gilgeous-Alexander", "Team": "Oklahoma City Thunder", "Position": "PG"},
    {"Player": "Nikola Jokic", "Team": "Denver Nuggets", "Position": "C"},
    {"Player": "Jamal Murray", "Team": "Denver Nuggets", "Position": "PG"},
    {"Player": "Norman Powell", "Team": "Miami Heat", "Position": "SG"},
    {"Player": "Alperen Sengun", "Team": "Houston Rockets", "Position": "C"},
    {"Player": "Pascal Siakam", "Team": "Indiana Pacers", "Position": "PF"},
    {"Player": "Karl-Anthony Towns", "Team": "New York Knicks", "Position": "C"},
    {"Player": "Victor Wembanyama", "Team": "San Antonio Spurs", "Position": "C"},
    {"Player": "Giannis Antetokounmpo", "Team": "Milwaukee Bucks", "Position": "PF"}
]

df_2026 = pd.DataFrame(all_stars_2026_data)
df_2026['Year'] = 2026

# Reorder columns to match existing df
df_2026 = df_2026[['Year', 'Player', 'Team', 'Position']]

# Concatenate with the scraped data
df_final = pd.concat([df, df_2026], ignore_index=True)

# Sort again including 2026
df_final = df_final.sort_values(by=['Year', 'Player'], ascending=[False, True])

print(f"Added {len(df_2026)} players from 2026.")
print(f"Total rows: {len(df_final)}")
print(df_final.head())

# Save updated DataFrame
try:
    df_final.to_csv('data/nba_all_stars.csv', index=False)
    print("Updated data/nba_all_stars.csv with 2026 data.")
    # Update the 'df' variable so subsequent cells us the new data
    df = df_final
except PermissionError:
    print("ERROR: Could not save to data/nba_all_stars.csv because it is open in another program.")
except Exception as e:
    print(f"Error saving file: {e}")

Added 28 players from 2026.
Total rows: 193
     Year           Player                    Team    Position
188  2026   Alperen Sengun         Houston Rockets  Frontcourt
169  2026  Anthony Edwards  Minnesota Timberwolves       Guard
177  2026   Brandon Ingram         Toronto Raptors  Frontcourt
167  2026  Cade Cunningham         Detroit Pistons       Guard
170  2026    Chet Holmgren   Oklahoma City Thunder  Frontcourt
Updated data/nba_all_stars.csv with 2026 data.


In [5]:
# Preview the dataframe
df.head()

,Year,Player,Team,Position
188,2026,Alperen Sengun,Houston Rockets,Frontcourt
169,2026,Anthony Edwards,Minnesota Timberwolves,Guard
177,2026,Brandon Ingram,Toronto Raptors,Frontcourt
167,2026,Cade Cunningham,Detroit Pistons,Guard
170,2026,Chet Holmgren,Oklahoma City Thunder,Frontcourt
